# Human Review — NZ Moment Tensor Catalogue

> **Made by Claude. Not tested.** This notebook is indicative — it was
> generated to hand over a working recipe, but no cell has been executed
> end-to-end. Expect to troubleshoot: read error messages, check paths,
> and ask questions. Fixing it is part of the learning.

**Audience**: final-year undergraduate and masters students.

**What you will do**: pick an automated moment tensor solution from this
repository's `events/` archive, understand how it was made, re-run the
inversion yourself, tweak it (station selection, filter band, depth
range), decide whether you can do better than the machine — and publish
your reviewed solution into the **human catalogue** (`events_human/`),
which grows gradually as events get human eyes on them.

**Ground rules**
1. The automated archive `events/` is **read-only** for you. Never edit,
   delete, or commit anything inside it. Your work goes in
   `events_human/` only.
2. Never run the publishing scripts (`run03_publish.py`, anything that
   sends email). Review is offline.
3. Every solution you publish must carry your name and your reasoning.

**Reading before you start** (all in this repository):
- `docs/METHOD.md` — how the automated pipeline works, step by step.
- `docs/REVIEW_LEARNINGS.md` — the reviewer watch-list: known failure
  modes (grid-edge depths, layer-boundary VR spikes, noise fitting,
  coda contamination) and how to spot them. **Read this twice.**
- Ristau (2008), SRL 79(3) — the NZ velocity models and regional MT
  practice this pipeline follows.
- The focal mechanism explorer at https://eq.comoglu.com/bb is great for
  building intuition about beachballs.


## 0. Installation (one-time, ~30 min)

Work in a terminal first; come back to the notebook when everything is
installed.

```bash
# 1. Fork the repository on GitHub (button top-right), then:
git clone https://github.com/<YOUR-USERNAME>/auto_tdmt_NZ.git
cd auto_tdmt_NZ

# 2. Install pixi (package manager) if you don't have it:
curl -fsSL https://pixi.sh/install.sh | bash
# then restart your terminal

# 3. Install the project environment (Python, ObsPy, mttime, ...):
pixi install

# 4. Add Jupyter to the environment:
pixi add notebook ipywidgets

# 5. Green's functions: download the tarball attached to the
#    "gf-latest" release on the ORIGINAL repository
#    (github.com/Dani-Lindsay/auto_tdmt_NZ -> Releases -> gf-latest),
#    and extract it somewhere, e.g.:
mkdir -p ~/mt_review/gf_library
tar -xzf ~/Downloads/<the-gf-tarball>.tar.gz -C ~/mt_review/gf_library
# The extracted tree should contain one directory per velocity model
# (nz_south_ristau2008, nz_north_ristau2008) with depth subdirectories.
# If the tarball extracts an extra top-level folder, point GF_DIR below
# at whichever directory directly contains the model directories.

# 6. Launch this notebook:
pixi run jupyter notebook human_review.ipynb
```

You do **not** need to build CPS or compute Green's functions — they are
precomputed. Waveforms download automatically from GeoNet (data licence
CC BY 3.0 NZ — credit GeoNet in anything you produce).


In [ ]:
# ============================================================
# 1. STUDENT CONFIGURATION — edit these, then run this cell first
# ============================================================
REVIEWER = "Your Name <you@university.ac.nz>"   # <-- EDIT
EVENT_ID = "2026p283255"                        # <-- EDIT: event to review
GF_DIR = "~/mt_review/gf_library"               # <-- EDIT: extracted GFs
WORKDIR = "~/mt_review/work"                    # scratch space (not in git)

import os, sys, json, shutil, warnings
from pathlib import Path
from datetime import date

GF_DIR = str(Path(GF_DIR).expanduser())
WORKDIR = Path(WORKDIR).expanduser() / EVENT_ID
WORKDIR.mkdir(parents=True, exist_ok=True)

# environment BEFORE importing the pipeline (paths are read at import)
os.environ["AUTO_TDMT_GF"] = GF_DIR
os.environ["AUTO_TDMT_EVENTS"] = str(WORKDIR / "auto_rerun")

REPO = Path.cwd()
assert (REPO / "config.py").exists(), \
    "Run the notebook from the auto_tdmt_NZ repository root"
sys.path.insert(0, str(REPO))
warnings.filterwarnings("ignore")

import config, waveforms, greens, invert, trigger
from geonet import get_event

assert Path(GF_DIR).exists(), f"GF library not found at {GF_DIR}"
for m in config.GF_MODELS:
    assert (Path(GF_DIR) / m).exists(), \
        f"model {m} missing under {GF_DIR} — check the tarball extraction"
print("environment OK — GF models:", config.GF_MODELS)


## 2. Browse the automated catalogue

Every automated solution lives in `events/<id>_Mw..._..km_.../` with a
`solution.json` (full provenance: every station used and dropped, with
reasons), the figures, and a draft email. The catalogue CSV is the
overview. Pick events that look *wrong* — grade C/D with red flags from
`REVIEW_LEARNINGS.md` — those are where a human adds the most value.


In [ ]:
import pandas as pd
cat = pd.read_csv(REPO / "events" / "catalogue.csv")
print(f"{len(cat)} automated solutions; grades:",
      cat.Grade.value_counts().to_dict())
cat[["PublicID", "Date", "Mw", "Depth", "VR", "DC", "Grade",
     "NS", "AzGap", "quality_flag"]]


## 3. Load the automated solution for your event

Read the machine's answer *and its reasoning* before touching anything:
which stations did it drop, and why? Look at the figures — especially
`*_station_waveforms_*.jpg` (every candidate station with its fate) and
`*_depth_sensitivity.jpg` (check the depth is a plateau, not a spike —
see the layer-interface warning in REVIEW_LEARNINGS).


In [ ]:
from IPython.display import Image, display

auto_dir = config.find_event_dir(EVENT_ID, REPO / "events")
assert auto_dir, f"{EVENT_ID} not found in events/"
auto = json.loads((auto_dir / "solution.json").read_text())
p, q = auto["preferred"], auto["quality"]
print(f"AUTOMATED: Mw {p['mw']:.2f}  depth {p['depth_km']:g} km  "
      f"VR {p['vr']:.1f}%  DC {p['pdc']:.0f}%  grade {q['grade']}  "
      f"gap {q['azimuthal_gap_deg']:.0f} deg  band {auto.get('chosen_band')}")
print("plane1 %(strike).0f/%(dip).0f/%(rake).0f" % p["plane1"])
print("used:", [r["station"] for r in auto["stations_used"]])
print("\ndropped (station | reason):")
for d in auto["stations_dropped"]:
    print("  ", d["station"], "|", d["reason"])
for jpg in sorted(auto_dir.glob("*.jpg")):
    display(Image(str(jpg), width=900))


## 4. Reproduce the baseline locally

This re-runs the full automated pipeline for your event into your
scratch space (nothing in the repo is touched). First run downloads
waveforms from GeoNet (a few minutes); re-runs use the local cache.


In [ ]:
from run02_process import process_event
baseline = process_event(EVENT_ID)
bp = baseline["preferred"]
print(f"\nBASELINE reproduced: Mw {bp['mw']:.2f} depth {bp['depth_km']:g} "
      f"VR {bp['vr']:.1f} DC {bp['pdc']:.0f}")


## 5. Your inversion sandbox

`run_custom()` lets you control what the automation decided for you:

- `keep` / `drop` — force the station set (codes like `"WEL"`)
- `band` — filter band in Hz, e.g. `(0.02, 0.1)` is 10–50 s
- `depths` — restrict the depth search, e.g. `[2, 3, 4, 5, 6, 8, 10]`

It returns the mttime inversion object and prints the preferred
solution. `compare()` gives the mechanism rotation between two runs
(plane-flip independent). Iterate: drop a suspect station, restrict the
depth to the physical plateau, try the other band — and watch what VR,
DC and the mechanism do. Rules of thumb are in REVIEW_LEARNINGS §2.


In [ ]:
EV = get_event(EVENT_ID)
MODEL = config.model_for_event(EV)

def run_custom(keep=None, drop=(), band=(0.02, 0.10), depths=None,
               tag="custom"):
    wd = WORKDIR / f"{tag}_{config.band_tag(band)}"
    wd.mkdir(parents=True, exist_ok=True)
    rows, dropped = waveforms.fetch_and_process(EV, wd, band)
    rows = [r for r in rows if r["station"] not in set(drop)
            and (keep is None or r["station"] in set(keep))]
    assert rows, "no stations left after keep/drop filtering"
    rows.sort(key=lambda r: r["distance_km"])
    dd = depths or invert.search_depths(EV, MODEL)
    greens.stage_event_greens(MODEL, rows, dd, band, wd / "greens")
    cwd = os.getcwd(); os.chdir(wd)
    try:
        inv = invert.run_inversion(
            invert.write_mtinv(EV, rows, dd, wd, wd / "greens"))
        inv.plot(view="waveform", option="preferred", format="jpg",
                 show=False)
    finally:
        os.chdir(cwd)
    mt = inv.moment_tensors[inv.preferred_tensor_id]
    sdr = tuple(float(x) for x in mt.fps[0][:3])
    print(f"[{tag}] {len(rows)} sta  depth {float(mt.depth):g} km  "
          f"Mw {mt.mw:.2f}  VR {float(mt.total_VR):.1f}  "
          f"DC {float(mt.pdc):.0f}  plane1 "
          f"{sdr[0]:.0f}/{sdr[1]:.0f}/{sdr[2]:.0f}")
    for r in mt.station_table.itertuples():
        print(f"    {r.station}: station VR {float(r.VR):.0f}")
    return inv, rows, dropped, wd

def preferred(inv):
    return inv.moment_tensors[inv.preferred_tensor_id]

def compare(inv_a, inv_b):
    a = tuple(float(x) for x in preferred(inv_a).fps[0][:3])
    b = tuple(float(x) for x in preferred(inv_b).fps[0][:3])
    ang = invert.tensor_angle_deg(a, b)
    print(f"mechanism rotation between runs: {ang:.1f} deg")
    return ang

def depth_profile(inv):
    import matplotlib.pyplot as plt
    zz = [float(mt.depth) for mt in inv.moment_tensors]
    vr = [float(mt.total_VR) for mt in inv.moment_tensors]
    dc = [float(mt.pdc) for mt in inv.moment_tensors]
    fig, ax = plt.subplots(1, 2, figsize=(10, 3))
    ax[0].plot(zz, vr, "ko-"); ax[0].set_xlabel("depth km")
    ax[0].set_ylabel("VR %")
    ax[1].plot(zz, dc, "ko-"); ax[1].set_xlabel("depth km")
    ax[1].set_ylabel("DC %")
    plt.tight_layout(); plt.show()

print("sandbox ready")


In [ ]:
# --- worked example: reproduce, then drop one station -------------
inv0, rows0, dropped0, wd0 = run_custom(tag="asis")
depth_profile(inv0)

# now YOUR tweak (edit and re-run as many times as you like):
inv1, rows1, dropped1, wd1 = run_custom(drop=["XXX"], tag="tweak1")
compare(inv0, inv1)


## 6. Decide and document

Fill in the review record honestly. `decision` is one of:
- `"accept_automated"` — the machine's solution stands; your review adds
  the human confirmation.
- `"revised"` — your tweaked solution replaces it in the human
  catalogue; say exactly what you changed and *why* (which watch-list
  item applied).
- `"reject"` — no defensible solution exists (e.g. no coherent signal);
  the record documents that finding.


In [ ]:
REVIEW = {
    "reviewer": REVIEWER,
    "date": str(date.today()),
    "decision": "revised",            # accept_automated | revised | reject
    "final_run_tag": "tweak1",        # which run_custom tag is final
    "changes": "dropped XXX (noise, station VR -40); "
               "depth restricted to 2-10 km plateau",
    "notes": "Watch-list items applied: grid-edge depth, passenger "
             "station. Mechanism stable (rotation 6 deg from baseline).",
}
FINAL_INV, FINAL_ROWS, FINAL_DROPPED, FINAL_WD = inv1, rows1, dropped1, wd1
print(json.dumps(REVIEW, indent=2))


## 7. Build your human solution and stage it in `events_human/`

This creates `events_human/<same-directory-name>/` containing your
`solution.json` (with a `human_review` block), your waveform-fit figure,
and copies of the automated figures for comparison, then appends a row
to `events_human/catalogue_human.csv`. Nothing in `events/` is touched.


In [ ]:
human = invert.summarize(FINAL_INV, EV, FINAL_ROWS, FINAL_DROPPED,
                         MODEL)
human["quality"] = invert.quality_gates(human)
human["human_review"] = REVIEW
human["automated_reference"] = {
    "solution_dir": auto_dir.name,
    "mw": p["mw"], "depth_km": p["depth_km"], "vr": p["vr"],
    "pdc": p["pdc"], "grade": q["grade"],
}

hdir = REPO / "events_human" / auto_dir.name
hdir.mkdir(parents=True, exist_ok=True)
(hdir / "solution.json").write_text(json.dumps(human, indent=2))
for src in sorted(FINAL_WD.glob("bbwaves*.jpg")):
    shutil.copy(src, hdir / f"{EVENT_ID}_human_waveform_fits.jpg")
for src in sorted(auto_dir.glob("*.jpg")):
    shutil.copy(src, hdir / f"auto_{src.name}")

hp, hq = human["preferred"], human["quality"]
row = {
    "PublicID": EVENT_ID, "Reviewer": REVIEWER,
    "ReviewDate": REVIEW["date"], "Decision": REVIEW["decision"],
    "Mw": round(hp["mw"], 2), "Depth": hp["depth_km"],
    "VR": round(hp["vr"], 1), "DC": round(hp["pdc"], 0),
    "Grade": hq["grade"],
    "strike1": round(hp["plane1"]["strike"]),
    "dip1": round(hp["plane1"]["dip"]),
    "rake1": round(hp["plane1"]["rake"]),
    "NS": hq["n_stations_used"],
    "Auto_Mw": p["mw"], "Auto_Depth": p["depth_km"],
    "Auto_Grade": q["grade"],
    "Changes": REVIEW["changes"],
}
cat_path = REPO / "events_human" / "catalogue_human.csv"
hcat = (pd.read_csv(cat_path) if cat_path.exists()
        else pd.DataFrame())
hcat = hcat[hcat.get("PublicID", pd.Series(dtype=str)) != EVENT_ID] \
    if len(hcat) else hcat
hcat = pd.concat([hcat, pd.DataFrame([row])], ignore_index=True)
hcat.to_csv(cat_path, index=False)
print("staged:", hdir)
print(f"human catalogue now has {len(hcat)} reviewed events")


## 8. Push your review to GitHub

Your changes must touch **only** `events_human/`. From a terminal in the
repository root:

```bash
git checkout -b review/<eventID>-<yourname>
git status               # confirm ONLY events_human/ files are listed
git add events_human/
git commit -m "human review <eventID>: <one-line summary> (<your name>)"
git push -u origin review/<eventID>-<yourname>
```

Then open a **Pull Request** on GitHub from your fork/branch to the main
repository. In the PR description paste your `REVIEW` block and one
sentence on what the automated pipeline should learn from this event.
The maintainer reviews and merges — that's your solution entering the
human catalogue. A PR that touches anything outside `events_human/`
will be closed unmerged.

## Troubleshooting (start here before asking)

- **Import errors** → you're not running from the repo root, or
  `pixi install` didn't finish.
- **GF assertion** → `GF_DIR` doesn't point at the directory containing
  the model folders; check the tarball extraction depth.
- **`No data available`** on download → that station has no data for
  this event; it's normal, the pipeline records it and moves on.
- **Everything fits terribly** → read `docs/REVIEW_LEARNINGS.md` §1–2;
  is your event offshore/no-signal, coda-contaminated, or riding a
  grid-edge depth? "No defensible solution" is a valid review outcome.

## Credits

Inversion: **mttime** (Chiang, LLNL). Green's functions: **CPS**
(Herrmann 2013) with the **Ristau (2008)** NZ velocity models
(doi:10.1785/gssrl.79.3.400). Waveforms and event data: **GeoNet**
(CC BY 3.0 NZ). Method lineage: Dreger & Helmberger (1993), Dreger
(2003), Minson & Dreger (2008). Cite them, not this repository.

> **Made by Claude. Not tested.** — you were warned at the top; now you
> know enough to fix it.
